<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Blackwell tcgen05 GEMM with contextvars

The `03_blackwell_mma` notebook multiplied a single 128×128 tile. Here we scale that kernel into a
full `C = A @ B` over an arbitrary **M×N×K** problem, keeping the one-function-per-warp-role shape
that made the single-tile kernel readable.

Two ideas carry the notebook:

- **A 2-D grid of CTAs does the tiling.** Each CTA owns one 128×128 output tile, reads its
  `(m, n)` tile coordinate from `cute.arch.block_idx()`, and walks the **K dimension** one 64-wide
  tile at a time, accumulating in TMEM. One pair of *tile-sized* TMA descriptors serves the whole
  grid: the per-CTA *coordinate*, not the descriptor, selects which slice of A and B to fetch.
- **Roles are still just Python.** Each warp picks its job with a plain `if warp_id == … elif …`
  and runs its own branch. The light producer warp caps its register file with a
  `with warp_registers(...)` block, so the heavy consumer and epilogue warps get a larger budget.

**You'll learn:** how a grid of CTAs tiles a large GEMM (one 128×128 tile per CTA, addressed
by `cute.arch.block_idx()`); how one tile-sized TMA descriptor serves every CTA by steering
*coordinates* instead of rebuilding descriptors; how a K-loop accumulates across K-tiles in TMEM;
and the minimal producer↔consumer `full`/`empty` mbarrier handshake that lets a single SMEM buffer
be reused for every K-tile. The structure stays modular — one `@cute.jit` subroutine per role,
selected by a plain `if`/`elif` on the warp id — and a `with warp_registers(...)` block scopes the
producer warp's register budget.

**Runs on:** **datacenter Blackwell `sm_100a` only** (B100/B200) — `tcgen05`/TMEM are not on
Hopper or consumer `sm_120`. No GPU? It compiles under dryrun (`CUTE_DSL_ARCH=sm_100a`).

In [ ]:
import contextlib

import cutlass
import cutlass.cute as cute
from cutlass.experimental import primitives as prims  # tcgen05 / mbarrier / TMA / barrier intrinsics + descriptors
import cutlass.experimental.cuda as cuda
import torch

## 1. Grid tiling, the K-loop, and the SMEM handshake

The launch is a 2-D grid of `(N/128, M/128)` CTAs. Each CTA reads its tile coordinate from
`cute.arch.block_idx()` (`grid.x` → N tile, `grid.y` → M tile) and turns it into global row/column
offsets `(m_off, n_off)`. Inside the CTA the warps keep their roles:

| warp(s) | role | per-CTA work |
|---|---|---|
| 0 | TMA producer | per K-tile, TMA-copies A[m_off, k] and B[k, n_off] global → SMEM |
| 1 | MMA consumer | per K-tile, runs `tcgen05_mma`; accumulates into the **same** TMEM tile |
| 2, 3 | exit | skipped, so the epilogue warps line up as a clean warp-group |
| 4-7 | epilogue | read the 128×128 result from TMEM and write C[m_off, n_off] |

The new ingredient is the **K-loop** and its **single-buffer handshake**. We keep just one 128×64
SMEM tile per operand and reuse it for every K-tile, so the producer must not overwrite a tile the
consumer is still reading. Two mbarriers chain them into a ping-pong:

| mbarrier | direction | meaning |
|---|---|---|
| `mbar_full` | producer → consumer | "this K-tile is loaded, go." |
| `mbar_empty` | consumer → producer | "I'm done with the buffer, refill it." Pre-signalled once before the loop so the first load doesn't deadlock on a consumer that hasn't run. |

With exactly one buffer, the handshake alternates between two mbarrier *phases*, so each side waits
on parity `kt % 2`. The consumer accumulates with `scale_d = kt != 0`: the first MMA overwrites
TMEM (no memset needed), every later one adds to it. When the K reduction finishes, `mbar_mma`
signals the epilogue. The TMA *coordinates* are `(k_off, m_off)` for A and `(k_off, n_off)` for B —
both operands are stored K-major, so K leads.

> **Metaprogramming aside.** `warp_registers` (next cell) is an ordinary Python **context manager** -- it scopes a *policy* with a `with` block, no GPU state involved. The same Python tool scales to launch configuration: a module-level `contextvars.ContextVar` set by a `with launch_config(block=...)` block lets a caller pick the launch shape without threading it through every call. Both keep the kernel body clean by moving configuration into plain Python.

In [ ]:
# Output tile is 128×128; K is consumed 64 elements at a time (64 fp16 = 128 B,
# which is exactly the s128b swizzle box).
TILE_M = 128
TILE_N = 128
TILE_K = 64


@contextlib.contextmanager
def warp_registers(regs, action=prims.SetMaxRegisterAction.DECREASE):
    """Scope a per-warp register budget with a `with` block.

    This is pure register *policy* (`setmaxregister`) — it changes how many
    registers the warp may use, not which threads run. The surrounding
    `if warp_id == ...` already selects the warp; this block only sets the budget.
    """
    prims.setmaxregister(regs, action)
    yield


# =============================================================================
# Warp workers: one @cute.jit subroutine per role.
# =============================================================================
@cute.jit
def tma_producer(
    smem_a,
    smem_b,
    tma_desc_a,
    tma_desc_b,
    mbar_full,
    mbar_empty,
    m_off,
    n_off,
    num_k_tiles,
):
    # One elected thread drives the whole warp's TMA traffic.
    prims.bar_warp_sync(cute.arch.FULL_MASK)
    if prims.elect_sync():
        size_a = (TILE_K * TILE_M) * cutlass.Float16.width // 8
        size_b = (TILE_K * TILE_N) * cutlass.Float16.width // 8
        for kt in range(num_k_tiles):
            # Step 1. Wait until the consumer has freed the SMEM buffer.
            # `empty` is pre-signalled (phase 1) and the consumer flips it once
            # per K-tile, so the phase to wait on before iteration kt is kt % 2 —
            # which lets the first iteration pass straight through.
            while not prims.mbarrier_try_wait_parity(mbar_empty, kt % 2):
                pass
            # Step 2. Fire the two bulk TMA copies for this CTA's (m, n) tile.
            # A is (M, K), B is (K, N) — both K-major — so the TMA coordinate
            # leads with the K offset.
            k_off = kt * TILE_K
            prims.mbarrier_arrive_expect_tx(mbar_full, size_a + size_b)
            prims.cp_async_bulk_tensor_shared_cta_global(
                smem_a, tma_desc_a.get_ptr(), (k_off, m_off), mbar_full
            )
            prims.cp_async_bulk_tensor_shared_cta_global(
                smem_b, tma_desc_b.get_ptr(), (k_off, n_off), mbar_full
            )


@cute.jit
def mma_consumer(
    smem_a, smem_b, tmem_ptr, mbar_full, mbar_empty, mbar_mma, num_k_tiles
):
    # One elected thread issues the MMAs for the whole warp.
    prims.bar_warp_sync(cute.arch.FULL_MASK)
    if prims.elect_sync():
        idesc = prims.Tcgen05InstrDesc.build(
            c_dtype=cutlass.Float32, n_dim=TILE_N, m_dim=TILE_M
        )
        for kt in range(num_k_tiles):
            # Step 1. Wait for the producer's data for this K-tile.
            while not prims.mbarrier_try_wait_parity(mbar_full, kt % 2):
                pass
            # Step 2. Run tcgen05 MMA into TMEM, accumulating across K.
            desc_a = prims.Tcgen05SmemDesc.build(
                smem_a,
                leading_byte_offset=16,
                stride_byte_offset=1024,
                layout=prims.Tcgen05SmemSwizzle.SWIZZLE_128B,
            )
            desc_b = prims.Tcgen05SmemDesc.build(
                smem_b,
                leading_byte_offset=16,
                stride_byte_offset=1024,
                layout=prims.Tcgen05SmemSwizzle.SWIZZLE_128B,
            )
            # Only the first MMA of the whole K-loop overwrites TMEM; the rest
            # accumulate on top. `kt` is a staged int, so compare against 0.
            scale_d = kt != 0
            # fp16 tensor-core K is 16, so 4 steps cover one 64-wide K-tile.
            for i in cutlass.range_constexpr(4):
                off: cutlass.Constexpr[int] = 16 * 2 * i
                prims.tcgen05_mma(
                    prims.Tcgen05MMAKind.F16,
                    prims.CTAGroup.CTA_1,
                    tmem_ptr,
                    desc_a.advance_start_address(off),
                    desc_b.advance_start_address(off),
                    idesc,
                    scale_d,
                )
                scale_d = True
            # Step 3. Release the SMEM buffer back to the producer.
            prims.tcgen05_commit(mbar_empty)
        # Step 4. Whole accumulation done: wake the epilogue.
        prims.tcgen05_commit(mbar_mma)


@cute.jit
def epilogue(matrix_c, tmem_ptr_i32, mbar_mma, thread_id, warp_id, m_off, n_off, tmem_num_col):
    # Step 1. Wait for the MMA to finish.
    while not prims.mbarrier_try_wait_parity(mbar_mma, 0):
        pass
    # Step 2. Copy the TMEM accumulator out to this CTA's 128×128 tile of C
    # at global offset (m_off, n_off).
    warpid_in_epi_wg = warp_id % 4
    tmem_base = prims.TmemAddr(tmem_ptr_i32.load())
    row_id = tmem_base.row_id + warpid_in_epi_wg * 32
    for n in range(0, tmem_num_col, 2):
        tmem = prims.TmemAddr.from_row_col(row_id, tmem_base.col_id + n).as_ptr(
            cutlass.Float32
        )
        c_rmem = prims.tcgen05_ld("32x32b", tmem, num=2)
        m_id = thread_id % 128
        for i in cutlass.range_constexpr(2):
            matrix_c[m_off + m_id, n_off + n + i] = cutlass.Float16(c_rmem[i])


# =============================================================================
# Kernel: set up shared state, then dispatch each warp to its worker.
# =============================================================================
@cute.kernel
def gemm_kernel(
    tma_desc_a: cutlass.GridConstant[cuda.TensorMap],
    tma_desc_b: cutlass.GridConstant[cuda.TensorMap],
    matrix_c: cutlass.Array,
    problem_size: cutlass.Constexpr,
) -> None:
    # Step 1. Read this CTA's output tile from its block index: grid.x indexes
    # N tiles, grid.y indexes M tiles.
    M, K, N = problem_size
    num_k_tiles: cutlass.Constexpr = K // TILE_K
    bid_n, bid_m, _ = cute.arch.block_idx()
    m_off = bid_m * TILE_M
    n_off = bid_n * TILE_N
    thread_id, _, _ = cute.arch.thread_idx()
    warp_id = cute.arch.warp_idx()
    tmem_num_col = 128

    # Step 2. Declare shared state: one 128×64 buffer per operand (reused across
    # K-tiles via the full/empty handshake), the three mbarriers, and a slot for
    # the TMEM pointer.
    smem_a = cutlass.Array(
        cutlass.Float16, (TILE_M, TILE_K), space=cutlass.AddressSpace.smem
    )
    smem_b = cutlass.Array(
        cutlass.Float16, (TILE_N, TILE_K), space=cutlass.AddressSpace.smem
    )
    mbar_full = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem)
    mbar_empty = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem)
    mbar_mma = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem)
    tmem_ptr_i32 = cutlass.Array(cutlass.Int32, 1, space=cutlass.AddressSpace.smem)

    # Step 3. One elected thread prefetches the descriptors and inits the
    # mbarriers. `empty` is pre-armed so the producer's first K-tile can proceed.
    if prims.elect_sync():
        prims.prefetch_tensormap(tma_desc_a.get_ptr())
        prims.prefetch_tensormap(tma_desc_b.get_ptr())
        prims.mbarrier_init(mbar_full, 1)
        prims.mbarrier_init(mbar_empty, 1)
        prims.mbarrier_init(mbar_mma, 1)
        prims.mbarrier_arrive(mbar_empty)  # pre-signal: buffer starts free
    prims.fence_mbarrier_init()
    cute.arch.barrier()

    # Step 4. Retire the unused warps. Warps 2-3 exit here, leaving 4-7 as a
    # clean warp-group for the epilogue. The CTA-wide barriers stay at kernel
    # scope, *between* the role `if` blocks, so every active warp reaches them.
    TMA_WARP, MMA_WARP = 0, 1

    if warp_id == 2 or warp_id == 3:
        prims.exit()

    # Step 5. The consumer warp owns the TMEM accumulator; allocate it before
    # anyone uses it.
    if warp_id == MMA_WARP:
        prims.tcgen05_alloc(tmem_ptr_i32, tmem_num_col)
    cute.arch.barrier()
    tmem_ptr = prims.make_tmem_ptr(tmem_ptr_i32.load(), cutlass.Int32)

    # Step 6. Dispatch each warp to its worker. Producer and consumer run
    # concurrently, synchronized by the SMEM full/empty mbarriers.
    if warp_id == TMA_WARP:
        # The producer is light, so cap its register budget for this scope. The
        # `if` above already selected the warp; the context manager only sets
        # policy, so it adds no gating.
        with warp_registers(40):
            tma_producer(
                smem_a,
                smem_b,
                tma_desc_a,
                tma_desc_b,
                mbar_full,
                mbar_empty,
                m_off,
                n_off,
                num_k_tiles,
            )
    elif warp_id == MMA_WARP:
        mma_consumer(
            smem_a, smem_b, tmem_ptr, mbar_full, mbar_empty, mbar_mma, num_k_tiles
        )
    elif warp_id == 4 or warp_id == 5 or warp_id == 6 or warp_id == 7:
        epilogue(
            matrix_c, tmem_ptr_i32, mbar_mma, thread_id, warp_id, m_off, n_off, tmem_num_col
        )

    # Step 7. Sync, then the consumer releases the TMEM it allocated.
    cute.arch.barrier()
    if warp_id == MMA_WARP:
        prims.tcgen05_dealloc(tmem_ptr, tmem_num_col)
        prims.tcgen05_relinquish_alloc_permit()

## 2. Host: one descriptor pair, a 2-D grid of tiles

The trick: the TMA descriptors span the **whole** A and B but carry a **tile-sized** 128×64 box.
That single pair covers every CTA — the per-CTA coordinate slides the box to the right tile, so we
never rebuild a descriptor per CTA. A is row-major `(M, K)`; B is K-major `(K, N)`, so its box and
TMA coordinates lead with K to match A. The launch is a 2-D grid `(N/128, M/128)` of 256-thread
(8-warp) blocks — one block per 128×128 output tile.

In [ ]:
# =============================================================================
# Host: build the A/B TMA descriptors and launch the multi-CTA GEMM kernel.
# =============================================================================
@cute.jit
def gemm(
    matrix_a: cutlass.Array,
    matrix_b: cutlass.Array,
    matrix_c: cutlass.Array,
    problem_size: cutlass.Constexpr,
) -> None:
    M, K, N = problem_size
    # Tile-sized descriptors over the WHOLE A and B: the box is one 128×64 tile,
    # and each CTA steers it with coordinates, so this single pair serves the grid.
    tma_desc_a = cuda.create_tensor_map_tiled_from_view(
        matrix_a, box_dims=(TILE_M, TILE_K), swizzle=cuda.TensorMapSwizzle.s128b
    )
    # B is stored K-major (shape (K, N), K contiguous), so its mode order is
    # (K, N): box and TMA coords lead with K, matching A's convention.
    tma_desc_b = cuda.create_tensor_map_tiled_from_view(
        matrix_b, box_dims=(TILE_K, TILE_N), swizzle=cuda.TensorMapSwizzle.s128b
    )
    # A 2-D grid of 128×128 output tiles: grid.x over N, grid.y over M.
    grid_n = N // TILE_N
    grid_m = M // TILE_M
    gemm_kernel(tma_desc_a, tma_desc_b, matrix_c, problem_size).launch(
        grid=(grid_n, grid_m, 1), block=(256, 1, 1)
    )

## 3. Run it and check against PyTorch

The driver runs `M = N = 256`, `K = 128` — a **2×2 grid** of 128×128 tiles, each CTA doing a
2-step K-loop. `b` is built transposed (`.T` flips only the *logical* shape, leaving the data
K-major), so it matches the descriptor's `(K, N)` layout. The PyTorch reference is computed on the
CPU, so the correctness check passes identically on real Blackwell hardware.

In [ ]:
# =============================================================================
# Main: build A/B/C, run the GEMM, and verify against PyTorch.
# =============================================================================
# Step 1. Build A/B/C. A is (M, K) row-major; B is (N, K) but stored K-major via
# .T. At 256×256×128 this is a 2×2 grid of 128×128 tiles, each CTA running a
# 2-step K-loop.
M, N, K = 256, 256, 128
a = torch.randn(M, K, dtype=torch.float16, device="cuda")
b = torch.randn(N, K, dtype=torch.float16, device="cuda").T  # K-major
c = torch.zeros(M, N, dtype=torch.float16, device="cuda")

# Step 2. Run the GEMM.
gemm(
    cute.runtime.from_dlpack(a),
    cute.runtime.from_dlpack(b),
    cute.runtime.from_dlpack(c),
    (M, K, N),
)

# Step 3. Verify. Reference on the CPU so the correctness check runs identically
# on real Blackwell hardware.
ref = a.cpu().float() @ b.cpu().float()
torch.testing.assert_close(c.cpu().float(), ref, atol=1e-2, rtol=1e-2)
print("PASS")

# Expected output:
# PASS

## Try it yourself

1. The grid is `(N/128, M/128)` and `bid_n, bid_m, _ = cute.arch.block_idx()`. Why is `grid.x`
   mapped to N and `grid.y` to M? What breaks if M or N is not a multiple of 128?
2. `mbar_empty` is pre-signalled once before the loop. Trace the `full`/`empty` parities across
   `kt = 0, 1, 2, …` — why must the producer wait on `kt % 2` (not `(kt+1) % 2`)?
3. `scale_d = kt != 0` makes only the first K-tile overwrite TMEM. What would memset-ing the
   accumulator instead cost, and why is "first write overwrites" cheaper for a long K-loop?
4. Every CTA loads its own A row-block and B col-block independently. Which loads do neighbouring
   CTAs *share*, and how would CTA-multicast TMA (`...shared_cluster_global`) exploit that?